In [1]:
from pathlib import Path
import json, math, time
from collections import Counter
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_ROOT = Path('/kaggle/input/datasets/robikscube/driving-video-with-object-tracking')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('/kaggle/input')
OUT = Path('/kaggle/working/bdd100k_yolo')
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
N_TRAIN, N_VAL, N_TEST = 900, 50, 50
ALLOW_SMALLER_TRAIN = True  # Mettre True SEULEMENT si <1000 vidéos annotées; alors train = disponibles - 100
TRAIN_STRIDE = 10
ANNOTATED_FPS = 5.0
COORD_WIDTH, COORD_HEIGHT = 1280, 720
IMG_EXT = '.jpg'
JPEG_QUALITY = 90

# Règle à choisir après la vérification visuelle
SYNC_MODE = 'ratio'   # 'ratio' -> round(frameIndex * video_fps / 5); 'six' -> frameIndex * 6
SYNC_OFFSET = 0
SYNC_VERIFIED = True  # Passer à True UNIQUEMENT après avoir inspecté les boîtes sur plusieurs vidéos

CLASS_TO_ID = {'pedestrian': 0, 'car': 1, 'bus': 2, 'truck': 3}
CLASS_NAMES = ['pedestrian', 'car', 'bus', 'truck']

print('Racine Kaggle :', DATA_ROOT)
print('Sortie YOLO :', OUT)

Racine Kaggle : /kaggle/input/datasets/robikscube/driving-video-with-object-tracking
Sortie YOLO : /kaggle/working/bdd100k_yolo


### About dataset

In [2]:
candidates = list(DATA_ROOT.rglob('mot_labels.csv'))
assert len(candidates) >= 1, 'mot_labels.csv introuvable. Attachez le dataset dans Kaggle.'
CSV_PATH = candidates[0]
video_extensions = {'.mp4', '.mov', '.avi', '.mkv'}
video_list = sorted(
    p for p in DATA_ROOT.rglob('*')
    if p.is_file() and p.suffix.lower() in video_extensions
    and 'videos' in p.parts and 'train' in p.parts
)
video_map = {p.stem: p for p in video_list}
assert len(video_map) == len(video_list), 'Identifiants vidéo dupliqués dans plusieurs dossiers.'

wanted_columns = [
    'name', 'videoName', 'frameIndex', 'category', 'haveVideo',
    'box2d.x1', 'box2d.y1', 'box2d.x2', 'box2d.y2'
]
labels = pd.read_csv(CSV_PATH, usecols=wanted_columns, low_memory=False)
labels['videoName'] = labels['videoName'].astype('string').str.strip()
labels['category'] = labels['category'].astype('string').str.strip().str.lower()
labels['frameIndex'] = pd.to_numeric(labels['frameIndex'], errors='coerce')
labels = labels.loc[labels['videoName'].notna() & labels['frameIndex'].notna()].copy()
labels['frameIndex'] = labels['frameIndex'].astype('int32')
for col in ['box2d.x1','box2d.x2','box2d.y1','box2d.y2']:
    labels[col] = pd.to_numeric(labels[col], errors='coerce')
labels['target_box'] = (
    labels['category'].isin(CLASS_NAMES)
    & labels[['box2d.x1','box2d.x2','box2d.y1','box2d.y2']].notna().all(axis=1)
    & (labels['box2d.x2'] > labels['box2d.x1'])
    & (labels['box2d.y2'] > labels['box2d.y1'])
)
labels['flag_have_video'] = labels['haveVideo'].astype('string').str.strip().str.lower().eq('true')

actual_video_ids = set(video_map)
labels = labels.loc[labels['videoName'].isin(actual_video_ids)].copy()
frame_counts = labels.groupby('videoName')['frameIndex'].nunique()
target_counts = labels.loc[labels['target_box']].groupby('videoName').size()

# Exiger un ensemble de frames annotées suffisamment long, et au moins une boîte dans les quatre classes
eligible_ids = sorted(v for v in actual_video_ids if frame_counts.get(v,0) >= 20 and target_counts.get(v,0) > 0)
missing_ids = sorted(actual_video_ids - set(eligible_ids))

audit = pd.DataFrame({'videoName': sorted(actual_video_ids)})
audit['rows_csv'] = audit['videoName'].map(labels.groupby('videoName').size()).fillna(0).astype(int)
audit['frames_csv'] = audit['videoName'].map(frame_counts).fillna(0).astype(int)
audit['target_bbox'] = audit['videoName'].map(target_counts).fillna(0).astype(int)
audit['has_true_flag'] = audit['videoName'].isin(labels.loc[labels['flag_have_video'].fillna(False),'videoName'])
audit['eligible'] = audit['videoName'].isin(eligible_ids)
audit.to_csv(OUT/'video_eligibility_audit.csv',index=False)

print('Fichiers vidéo :',len(video_map))
print('Vidéos avec lignes CSV :',int((audit.rows_csv>0).sum()))
print('Vidéos avec annotations cibles :',int((audit.target_bbox>0).sum()))
print('Vidéos admissibles au split :',len(eligible_ids))
print('Vidéos sans haveVideo=True, mais admissibles :',int((audit.eligible & ~audit.has_true_flag).sum()))
print('Vidéos non admissibles :',len(missing_ids))

Fichiers vidéo : 1000
Vidéos avec lignes CSV : 961
Vidéos avec annotations cibles : 961
Vidéos admissibles au split : 961
Vidéos sans haveVideo=True, mais admissibles : 0
Vidéos non admissibles : 39


In [3]:
from sklearn.model_selection import train_test_split

# ==========================================
# 1. Vérifier les vidéos disponibles
# ==========================================

eligible_ids = sorted(set(eligible_ids))
n_available = len(eligible_ids)

N_TRAIN_REQUESTED = N_TRAIN

print("Vidéos disponibles :", n_available)
print(
    "Vidéos initialement demandées :",
    N_TRAIN + N_VAL + N_TEST
)

# Vérifier que val et test sont possibles
if n_available <= N_VAL + N_TEST:
    raise ValueError(
        f"Seulement {n_available} vidéos disponibles : "
        "impossible de constituer les trois partitions."
    )

# ==========================================
# 2. Adapter N_TRAIN automatiquement
# ==========================================

N_TRAIN = min(
    N_TRAIN_REQUESTED,
    n_available - N_VAL - N_TEST
)

n_required = N_TRAIN + N_VAL + N_TEST

print("\nRépartition choisie :")
print("Train :", N_TRAIN)
print("Val   :", N_VAL)
print("Test  :", N_TEST)
print("Total :", n_required)

# ==========================================
# 3. Sélection reproductible des vidéos
# ==========================================

rng = np.random.default_rng(SEED)

chosen_ids = sorted(
    rng.choice(
        eligible_ids,
        size=n_required,
        replace=False
    ).tolist()
)

# ==========================================
# 4. Identifier les vidéos contenant des bus
# ==========================================

bus_ids = set(
    labels.loc[
        labels["target_box"]
        & labels["category"].eq("bus"),
        "videoName"
    ]
)

strat = [int(v in bus_ids) for v in chosen_ids]

# ==========================================
# 5. Split train / évaluation
# ==========================================

train_ids, eval_ids = train_test_split(
    chosen_ids,
    train_size=N_TRAIN,
    test_size=N_VAL + N_TEST,
    random_state=SEED,
    stratify=strat
)

# ==========================================
# 6. Split validation / test
# ==========================================

eval_strat = [int(v in bus_ids) for v in eval_ids]

val_ids, test_ids = train_test_split(
    eval_ids,
    train_size=N_VAL,
    test_size=N_TEST,
    random_state=SEED + 1,
    stratify=eval_strat
)

sets = {
    "train": sorted(train_ids),
    "val": sorted(val_ids),
    "test": sorted(test_ids)
}

# ==========================================
# 7. Vérifier l'absence de chevauchement
# ==========================================

assert len(train_ids) == N_TRAIN
assert len(val_ids) == N_VAL
assert len(test_ids) == N_TEST

assert not (set(train_ids) & set(val_ids))
assert not (set(train_ids) & set(test_ids))
assert not (set(val_ids) & set(test_ids))

print("\n✅ Aucun chevauchement entre les partitions")

# ==========================================
# 8. Créer split_manifest.csv
# ==========================================

OUT.mkdir(parents=True, exist_ok=True)

split_df = pd.DataFrame([
    {
        "videoName": vid,
        "split": split,
        "video_path": str(video_map[vid]),
        "annotated_frames": int(frame_counts.get(vid, 0)),
        "n_target_bbox": int(target_counts.get(vid, 0)),
        "has_bus": vid in bus_ids
    }
    for split, vids in sets.items()
    for vid in vids
])

split_df.to_csv(
    OUT / "split_manifest.csv",
    index=False
)

print("\nNombre de vidéos par split :")
print(split_df.groupby("split")["videoName"].nunique())

# ==========================================
# 9. Vérifier les classes par partition
# ==========================================

target_split_df = labels.loc[
    labels["target_box"]
    & labels["videoName"].isin(chosen_ids)
].copy()

split_lookup = dict(
    zip(split_df.videoName, split_df.split)
)

target_split_df["split"] = (
    target_split_df["videoName"].map(split_lookup)
)

print("\nBoîtes cibles AVANT stride10 par split :")

display(
    pd.crosstab(
        target_split_df["split"],
        target_split_df["category"]
    )
)

# ==========================================
# 10. Sauvegarder la configuration
# ==========================================

meta = {
    "seed": SEED,
    "n_eligible": n_available,
    "n_train_requested": N_TRAIN_REQUESTED,
    "n_train": N_TRAIN,
    "n_val": N_VAL,
    "n_test": N_TEST,
    "train_stride": TRAIN_STRIDE,
    "annotated_fps": ANNOTATED_FPS,
    "class_to_id": CLASS_TO_ID,
    "csv_path": str(CSV_PATH),
    "policy": (
        "video-disjoint; "
        "bus-presence stratified; "
        "all annotated evaluation frames"
    )
}

(OUT / "split_config.json").write_text(
    json.dumps(meta, indent=2),
    encoding="utf8"
)

print("\n✅ Partition terminée")
print("Manifeste :", OUT / "split_manifest.csv")


Vidéos disponibles : 961
Vidéos initialement demandées : 1000

Répartition choisie :
Train : 861
Val   : 50
Test  : 50
Total : 961

✅ Aucun chevauchement entre les partitions

Nombre de vidéos par split :
split
test      50
train    861
val       50
Name: videoName, dtype: int64

Boîtes cibles AVANT stride10 par split :


category,bus,car,pedestrian,truck
split,,,,
test,2994,75851,16393,6180
train,33838,1322896,228269,83973
val,2022,74753,9788,5816



✅ Partition terminée
Manifeste : /kaggle/working/bdd100k_yolo/split_manifest.csv


### To yolo Format 

In [5]:
def mp4_index(frame_index, video_fps, mode='ratio', offset=0):
    if mode == 'ratio':
        return int(round(frame_index * video_fps / ANNOTATED_FPS)) + offset
    if mode == 'six':
        return frame_index * 6 + offset
    raise ValueError(mode)

def read_frame_at(cap, position):
    if position < 0:
        return None
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(position))
    ok, im = cap.read()
    return im if ok else None

def draw_annotation_overlay(im, frame_rows):
    im = im.copy()
    h,w = im.shape[:2]
    sx,sy = w / COORD_WIDTH, h / COORD_HEIGHT
    for _, row in frame_rows.iterrows():
        if not row['target_box']: continue
        x1 = int(row['box2d.x1'] * sx)
        y1 = int(row['box2d.y1'] * sy)
        x2 = int(row['box2d.x2'] * sx)
        y2 = int(row['box2d.y2'] * sy)
        cv2.rectangle(im,(x1,y1),(x2,y2),(0,255,0),2)
    return cv2.cvtColor(im,cv2.COLOR_BGR2RGB)

In [6]:
SYNC_MODE = "six"
SYNC_OFFSET = 0
SYNC_VERIFIED = True

assert SYNC_VERIFIED, (
    'Export bloqué : vérifiez les overlays de la cellule précédente, puis mettez SYNC_VERIFIED=True en cellule 1.'
)

OUT= Path("databdd")
sets = {'train':sorted(train_ids), 'val':sorted(val_ids)}


for split in ['train', 'val']:
    (OUT/'images'/split).mkdir(parents=True,exist_ok=True)
    (OUT/'labels'/split).mkdir(parents=True,exist_ok=True)

def bbox_yolo_rows(subframe, w, h):
    lines=[]
    for row in subframe.itertuples(index=False, name=None):
        cat=row[subframe.columns.get_loc('category')]
        if pd.isna(cat) or cat not in CLASS_TO_ID:
            continue
        vals=[row[subframe.columns.get_loc(k)] for k in
              ['box2d.x1','box2d.y1','box2d.x2','box2d.y2']]
        if not all(pd.notna(z) and np.isfinite(z) for z in vals):
            continue
        x1,y1,x2,y2=map(float,vals)
        x1,x2=np.clip([x1,x2],0,COORD_WIDTH)
        y1,y2=np.clip([y1,y2],0,COORD_HEIGHT)
        if x2<=x1 or y2<=y1:
            continue
        # Coordonnées des labels connues en résolution BDD100K 1280x720,
        # donc la normalisation est indépendante du redimensionnement éventuel.
        xc=(x1+x2)/(2*COORD_WIDTH)
        yc=(y1+y2)/(2*COORD_HEIGHT)
        bw=(x2-x1)/COORD_WIDTH
        bh=(y2-y1)/COORD_HEIGHT
        lines.append(f'{CLASS_TO_ID[cat]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
    return lines

frame_manifest=[]
errors=[]
labels_by_video=labels.groupby('videoName',sort=False)
start=time.time()
for split,vids in sets.items():
    print(f'Export {split}: {len(vids)} vidéos')
    for k,vid in enumerate(vids,1):
        ann=labels_by_video.get_group(vid)
        if split=='train':
            ann=ann.loc[ann.frameIndex.mod(TRAIN_STRIDE).eq(0)]
        frame_keys=np.sort(ann.frameIndex.unique())
        frame_groups={int(fi):grp for fi,grp in ann.groupby('frameIndex',sort=False)}
        cap=cv2.VideoCapture(str(video_map[vid]))
        if not cap.isOpened():
            errors.append({'split':split,'videoName':vid,'frameIndex':None,'problem':'video_open_failed'})
            continue
        fps=cap.get(cv2.CAP_PROP_FPS)
        total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        if not np.isfinite(fps) or fps<=0 or width<=0 or height<=0:
            cap.release()
            errors.append({'split':split,'videoName':vid,'frameIndex':None,'problem':'invalid_video_metadata'})
            continue
        last_idx=-1
        for fi in frame_keys:
            mp_idx=mp4_index(int(fi),fps,SYNC_MODE,SYNC_OFFSET)
            if mp_idx<0 or mp_idx>=total:
                errors.append({'split':split,'videoName':vid,'frameIndex':int(fi),
                               'mp4FrameIndex':mp_idx,'problem':'out_of_range'})
                continue
            # Pour frames proches (intervalle ~6), avancer en lecture séquentielle;
            # sinon faire un seek pour éviter le décodage de toutes les frames.
            if last_idx>=0 and 0 < mp_idx-last_idx <= 12:
                image=None
                good=True
                for _ in range(mp_idx-last_idx):
                    ok,image=cap.read()
                    if not ok:
                        good=False;break
                if not good: image=None
            else:
                image=read_frame_at(cap,mp_idx)
            last_idx=mp_idx
            if image is None:
                errors.append({'split':split,'videoName':vid,'frameIndex':int(fi),
                               'mp4FrameIndex':mp_idx,'problem':'decode_failed'})
                continue
            # Pas de ré-échantillonnage spatial dans les images exportées
            h,w=image.shape[:2]
            if (w,h)!=(COORD_WIDTH,COORD_HEIGHT):
                errors.append({'split':split,'videoName':vid,'frameIndex':int(fi),
                               'problem':f'unexpected_resolution_{w}x{h}'})
                continue
            stem=f'{vid}__{int(fi):06d}'
            image_path=OUT/'images'/split/(stem+IMG_EXT)
            label_path=OUT/'labels'/split/(stem+'.txt')
            if not cv2.imwrite(str(image_path),image,[cv2.IMWRITE_JPEG_QUALITY,JPEG_QUALITY]):
                errors.append({'split':split,'videoName':vid,'frameIndex':int(fi),'problem':'imwrite_failed'})
                continue
            yolo_lines=bbox_yolo_rows(frame_groups[int(fi)],w,h)
            label_path.write_text(('\n'.join(yolo_lines)+'\n') if yolo_lines else '',encoding='utf8')
            frame_manifest.append({'split':split,'videoName':vid,
                'annotationFrameIndex':int(fi),'decodedMp4FrameIndex':mp_idx,
                'fpsMp4':fps,'image_path':str(image_path),
                'label_path':str(label_path),'n_boxes':len(yolo_lines)})
        cap.release()
        if k%50==0 or k==len(vids):
            print(f'  {k}/{len(vids)} vidéos; {len(frame_manifest)} images exportées')

frames_export=pd.DataFrame(frame_manifest)
errors_export=pd.DataFrame(errors)
frames_export.to_csv(OUT/'frame_manifest.csv',index=False)
errors_export.to_csv(OUT/'export_errors.csv',index=False)
print('Export terminé en secondes :',round(time.time()-start,1))
print('Frames exportées :',len(frames_export))
print('Échecs :',len(errors_export))
display(frames_export.groupby('split').agg(
    frames=('image_path','size'), bbox=('n_boxes','sum'),
    videos=('videoName','nunique')
))


Export train: 861 vidéos
  50/861 vidéos; 1041 images exportées
  100/861 vidéos; 2090 images exportées
  150/861 vidéos; 3066 images exportées
  200/861 vidéos; 4112 images exportées
  250/861 vidéos; 5130 images exportées
  300/861 vidéos; 6163 images exportées
  350/861 vidéos; 7199 images exportées
  400/861 vidéos; 8236 images exportées
  450/861 vidéos; 9257 images exportées
  500/861 vidéos; 10255 images exportées
  550/861 vidéos; 11297 images exportées
  600/861 vidéos; 12333 images exportées
  650/861 vidéos; 13350 images exportées
  700/861 vidéos; 14359 images exportées
  750/861 vidéos; 15374 images exportées
  800/861 vidéos; 16410 images exportées
  850/861 vidéos; 17437 images exportées
  861/861 vidéos; 17658 images exportées
Export val: 50 vidéos
  50/50 vidéos; 27344 images exportées
Export terminé en secondes : 6129.2
Frames exportées : 27344
Échecs : 90


,frames,bbox,videos
split,,,
train,17658,172721,861
val,9686,92073,50


### Split Test 

In [8]:
import pandas as pd
import shutil
from pathlib import Path
from collections import Counter

# ==========================================
# 1. CONFIGURATION
# ==========================================

# Manifeste créé lors du split
MANIFEST_PATH = Path("/kaggle/working/bdd100k_yolo/split_manifest.csv")


# CSV original contenant les annotations MOT
SOURCE_CSV = Path(CSV_PATH)

# Nouveau dossier de sortie
DEST_ROOT = Path("/kaggle/working/bdd100k_test")
DEST_VIDEOS = DEST_ROOT / "test"
DEST_CSV = DEST_ROOT / "mot_labels.csv"

DEST_VIDEOS.mkdir(parents=True, exist_ok=True)

assert MANIFEST_PATH.exists(), "split_manifest.csv introuvable"
assert SOURCE_CSV.exists(), "mot_labels.csv original introuvable"

# ==========================================
# 2. RÉCUPÉRER UNIQUEMENT LES VIDÉOS TEST
# ==========================================

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"videoName": str}
)

test_df = manifest[
    manifest["split"] == "test"
].copy()

test_ids = set(test_df["videoName"])

print("Nombre de vidéos test :", len(test_ids))

assert len(test_ids) == 50, (
    f"50 vidéos attendues, {len(test_ids)} trouvées"
)

assert len(test_df) == 50, (
    "Le manifeste contient des vidéos test dupliquées"
)

# Vérifier les chemins d'origine
source_videos = [
    Path(p) for p in test_df["video_path"]
]

missing = [
    str(p) for p in source_videos
    if not p.is_file()
]

assert not missing, (
    f"{len(missing)} vidéos introuvables : {missing[:5]}"
)

# Vérifier les noms de fichiers
video_names = [p.name for p in source_videos]

assert len(video_names) == len(set(video_names)), (
    "Deux vidéos ont le même nom de fichier"
)

# Empêcher le mélange avec un ancien test
existing_videos = {
    p.name for p in DEST_VIDEOS.iterdir()
    if p.is_file()
}

extra_videos = existing_videos - set(video_names)

assert not extra_videos, (
    "Le dossier contient d'anciennes vidéos étrangères "
    f"au test actuel : {list(extra_videos)[:5]}"
)

# ==========================================
# 3. COPIER LES 50 VIDÉOS TEST
# ==========================================

print("\nCopie des vidéos test...")

for i, source in enumerate(source_videos, start=1):

    destination = DEST_VIDEOS / source.name

    if (
        not destination.exists()
        or destination.stat().st_size != source.stat().st_size
    ):
        shutil.copy2(source, destination)

    if i % 10 == 0:
        print(f"{i}/{len(source_videos)} vidéos copiées")

print("✅ Vidéos test copiées")

# ==========================================
# 4. EXTRAIRE LES ANNOTATIONS TEST
# ==========================================

print("\nExtraction des annotations MOT test...")

rows_total = 0
counts = Counter()
first_chunk = True

# Lecture par morceaux pour éviter de charger
# tout le fichier CSV en mémoire
for chunk in pd.read_csv(
    SOURCE_CSV,
    chunksize=150_000,
    dtype={"videoName": str}
):

    # Garder toutes les lignes des vidéos test
    subset = chunk[
        chunk["videoName"].isin(test_ids)
    ]

    subset.to_csv(
        DEST_CSV,
        mode="w" if first_chunk else "a",
        index=False,
        header=first_chunk
    )

    first_chunk = False

    rows_total += len(subset)

    counts.update(
        subset["videoName"].value_counts().to_dict()
    )

print("✅ CSV test créé")
print("Nombre de lignes conservées :", rows_total)

# ==========================================
# 5. VÉRIFICATIONS
# ==========================================

copied_videos = list(DEST_VIDEOS.glob("*.mp4"))

print("\n========== VÉRIFICATION ==========")

print("Vidéos test :", len(copied_videos))
print("Vidéos avec annotations :", len(counts))
print("Lignes dans le CSV :", rows_total)

Nombre de vidéos test : 50

Copie des vidéos test...
10/50 vidéos copiées
20/50 vidéos copiées
30/50 vidéos copiées
40/50 vidéos copiées
50/50 vidéos copiées
✅ Vidéos test copiées

Extraction des annotations MOT test...


/tmp/ipykernel_124/377506191.py:119: DtypeWarning: Columns (5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/tmp/ipykernel_124/377506191.py:119: DtypeWarning: Columns (5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/tmp/ipykernel_124/377506191.py:119: DtypeWarning: Columns (5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/tmp/ipykernel_124/377506191.py:119: DtypeWarning: Columns (5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/tmp/ipykernel_124/377506191.py:119: DtypeWarning: Columns (5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/tmp/ipykernel_124/377506191.py:119: DtypeWarning: Columns (5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in

✅ CSV test créé
Nombre de lignes conservées : 106003

========== VÉRIFICATION ==========
Vidéos test : 0
Vidéos avec annotations : 50
Lignes dans le CSV : 106003
